# Fine-tuning YOLO11n to count scaffold planks in photographs

**Abstract.** This notebook documents how we fine-tuned and evaluated a YOLO11n object detector that counts
steel scaffold planks in photographs of stacked planks. Only five labeled photographs are available, so we
evaluate the model by leave-one-image-out cross-validation: we hold out each photograph in turn, fine-tune a
detector on random crops of the other four, and compare its count on the held-out photograph with a manually
annotated count. We report counting error, localization recall and precision, and inference time. We also record
the export of the final model to Core ML, which failed in the recorded execution and succeeded after a dependency
fix in a second session (Section 8).

**Provenance of the recorded outputs.** The outputs below come from one execution on a Google Colab T4 GPU. In
that execution we ran an earlier revision of the code, before we added the comments and fixed the random seed, so
a new run gives slightly different numbers. We removed three kinds of output from this archived copy: per-epoch
training logs (condensed to the result lines), the figure of predictions (the photographs are confidential), and
the browser download widgets.

**Data confidentiality.** Ready to Bloom supplied the photographs for this task only. We do not store them in the
repository, so anyone who runs the notebook must upload them to the Colab session.

## 1. Experimental design

**Data.** The dataset contains five photographs of stacked steel scaffold planks viewed end-on. We marked every
plank end that is individually visible on the front face of any stack with one point. This gave 224 annotated
planks in total (54, 24, 73, 13 and 60 for the five photographs). We derived the training boxes from the points:
we set the box height from the median distance between neighboring points and the box width to five times the
height. These boxes are approximate and inherit the imprecision of the manual clicks.

**Model and training.** We fine-tuned YOLO11n (about 2.6 million parameters), pre-trained on COCO, for a single
class (plank end). Each training photograph contributed 60 random crops covering 45 to 100 percent of the image
side, resized to 640 pixels. Ultralytics applied further augmentation during training: rotation up to 10 degrees,
horizontal flips, scale variation of 0.4, mosaic, and HSV color jitter. We trained for 80 epochs with batch size
16. We disabled per-epoch validation and used the weights of the final epoch, so the held-out photograph could
not influence model selection.

**Evaluation protocol.** For each of the five photographs we trained a separate model on the other four and used
it to count the held-out photograph. We set the detection confidence threshold to 0.25 and the non-maximum
suppression IoU threshold to 0.5.

**Metrics.** The absolute count error is the absolute difference between the predicted and annotated counts, and
the mean absolute error (MAE) averages it over the photographs. Recall is the fraction of annotated planks whose
point lies inside at least one predicted box. Precision is the fraction of predicted boxes that contain at least
one annotated point. Inference time is the wall-clock time of one prediction after a warm-up call.

## 2. Computing environment

We query the GPU model and install the dependencies. We restrict numpy to releases below 2.4 because coremltools 9
failed to convert the model with numpy 2.4 (Section 8). We added this restriction after the recorded execution and
used it in the second session that produced the Core ML package.

In [1]:
# Report the GPU model and its memory (nvidia-smi is the NVIDIA system tool)
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Install the detector library, the AVIF image plugin, and the Core ML converter.
# The "numpy<2.4" pin keeps numpy below 2.4, because coremltools 9 fails with newer versions.
# The -q flag keeps the installation output short.
!pip -q install ultralytics pillow-avif-plugin coremltools "numpy<2.4"

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 3. Data loading

We upload the project archive (source code, the five photographs, and the annotation files) to the Colab session
and extract it. If Colab reports that a package changed during installation, we restart the session once before
this step.

In [2]:
# files is Colab's helper for moving files between the local computer and the session
from google.colab import files
# Open a file picker for the project archive colab_upload.zip
uploaded = files.upload()

Saving colab_upload.zip to colab_upload (1).zip


In [3]:
# Extract the archive into a folder named project (-q is quiet, -o overwrites old files)
!unzip -q -o colab_upload.zip -d project
# Move into the project folder so the relative paths used by the scripts work
%cd project
# List the code, image, and annotation folders to confirm that everything arrived
!ls src data/images data/annotations

/content/project
data/annotations:
plank1.json  plank2.json  plank3.json  plank4.json  plank5.json
data/images:
plank1.jpg  plank2.jpg	plank3.jpg  plank4.jpg	plank5.avif
src:
annotate.py	       eval_fastsam_matching.py  tune_fastsam.py
baseline_classical.py  export_coreml.py		 zero_shot_sam.py
common.py	       show_gt.py
count.py	       train_yolo.py


## 4. Leave-one-image-out evaluation

We train five detectors in sequence, one per held-out photograph. Each fold prints its result as one line. Training
took several minutes per fold on the T4 GPU.

In [4]:
# Train and evaluate the five folds, then print the mean absolute error.
# --epochs is the number of passes over the training crops, --crops the random crops per image.
!python src/train_yolo.py --epochs 80 --crops 60

{'image': 'plank1.jpg', 'predicted': 62, 'true': 54, 'abs_error': 8, 'recall': 1.0, 'precision': 0.9, 'ms': 11.4}
{'image': 'plank2.jpg', 'predicted': 22, 'true': 24, 'abs_error': 2, 'recall': 1.0, 'precision': 1.0, 'ms': 12.1}
{'image': 'plank3.jpg', 'predicted': 89, 'true': 73, 'abs_error': 16, 'recall': 0.92, 'precision': 0.79, 'ms': 11.0}
{'image': 'plank4.jpg', 'predicted': 19, 'true': 13, 'abs_error': 6, 'recall': 0.69, 'precision': 0.42, 'ms': 10.9}
{'image': 'plank5.avif', 'predicted': 35, 'true': 60, 'abs_error': 25, 'recall': 0.52, 'precision': 0.89, 'ms': 11.9}
Mean absolute error over 5 held-out images: 11.4 planks


## 5. Results

### 5.1 Counting accuracy

Table 1 lists, for each held-out photograph, the predicted and annotated counts, the absolute error, the
localization recall and precision, and the inference time on the T4 GPU.

In [5]:
# pandas reads and displays tables
import pandas as pd
# Read the per-image results saved by the training script
df = pd.read_csv("results/yolo_loo.csv")
# Show the table (Colab displays the value of the last expression in a cell)
df

,image,predicted,true,abs_error,recall,precision,ms
0,plank1.jpg,62,54,8,1.00,0.90,11.4
1,plank2.jpg,22,24,2,1.00,1.00,12.1
2,plank3.jpg,89,73,16,0.92,0.79,11.0
3,plank4.jpg,19,13,6,0.69,0.42,10.9
4,plank5.avif,35,60,25,0.52,0.89,11.9


The mean absolute error was 11.4 planks. Errors differed in sign and size across photographs. The detector
overcounted plank3 (89 predicted, 73 annotated) and plank1 (62, 54), and it undercounted plank5 (35, 60), where
recall was 0.52 because it missed most planks in the middle and right columns. Precision was lowest for plank4
(0.42), a photograph dominated by top surfaces of planks that resemble the target class. We read the recall of 1.00
on plank1 and plank2 with caution, because the point-in-box criterion is permissive when predicted boxes are large
or overlap. Inference took 11 to 12 ms per photograph.

### 5.2 Qualitative results

The next cell displays the held-out photographs with their predicted boxes. We do not include the figure in this
archived copy because the photographs are confidential. Running the cell regenerates it.

In [6]:
# OpenCV reads the overlay images
import cv2
# matplotlib draws the figure
import matplotlib.pyplot as plt
# Path finds the overlay files
from pathlib import Path

# Collect the saved overlay images, one per held-out photograph
paths = sorted(Path("results/overlays").glob("*_yolo_loo.jpg"))
# Create one panel per image in a single row
fig, axes = plt.subplots(1, len(paths), figsize=(5 * len(paths), 5))
# Fill each panel in turn
for ax, p in zip(axes, paths):
    # Read the image and convert BGR to RGB, the order matplotlib expects
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    # Title the panel with the photograph name
    ax.set_title(p.stem.replace("_yolo_loo", ""))
    # Hide the axis ticks
    ax.axis("off")
# Reduce the spacing between panels
plt.tight_layout()

## 6. Final model and inference time

We trained a final detector on all five photographs for deployment and export. Its counts are not a valid estimate
of generalization, because every photograph was also part of its training data. They show how closely the model
fits the training photographs, and we list them for completeness together with the inference time.

In [7]:
# Train one model on all five photographs (no held-out image)
!python src/train_yolo.py --final --epochs 80 --crops 60

final weights: runs/final_all/weights/last.pt


In [8]:
# Count planks in each photograph with the final model and print the time per image
!python src/count.py --weights runs/final_all/weights/last.pt

plank1.jpg: 59 planks  (18 ms)
plank2.jpg: 25 planks  (16 ms)
plank3.jpg: 73 planks  (12 ms)
plank4.jpg: 12 planks  (11 ms)
plank5.avif: 61 planks  (12 ms)


The final model counted 59, 25, 73, 12 and 61 planks against annotated counts of 54, 24, 73, 13 and 60. The
inference time was 11 to 18 ms per photograph on the T4 GPU.

## 7. Export to Core ML

We exported the final weights to Core ML so that the model can be used in an iOS application. The export runs on
Linux, but the exported model can only be executed and timed on a Mac or an iPhone, so we did not measure on-device
latency in this study.

In [9]:
# Convert the final weights to a Core ML package
!python src/export_coreml.py --weights runs/final_all/weights/last.pt
# List the weights folder to see whether the package was written
!ls runs/final_all/weights

requirements: Ultralytics requirement ['numpy>=1.14.5,<=2.3.5'] not found, attempting AutoUpdate...
 - numpy==2.4.6
 + numpy==2.3.5
Converting PyTorch Frontend ==> MIL Ops:   0% 0/716 [00:00<?, ? ops/s]ERROR:coremltools:
ERROR - converting 'int' op (located at: 'model/10/m/0/attn/479'):
ERROR ❌ CoreML: export failure 17.4s: only 0-dimensional arrays can be converted to Python scalars
TypeError: only 0-dimensional arrays can be converted to Python scalars
best.pt  last.pt


## 8. Core ML export: failure and resolution

**Failure in the recorded execution.** The converter raised a `TypeError` ("only 0-dimensional arrays can be
converted to Python scalars") at an integer cast operation inside the model, and it wrote no Core ML package. The
Colab environment had numpy 2.4.6, while Ultralytics requires numpy 2.3.5 or older. Ultralytics replaced numpy
automatically during the run, but the Python process had already loaded version 2.4.6, which is consistent with a
numpy version incompatibility being the cause.

**Resolution.** We repeated the export in a new Colab session with numpy restricted to versions below 2.4, using
the same final weights. It completed and produced a Core ML package of about 5.4 MB (5.3 MB of weights stored in
16-bit floating point). Inspection of the package specification shows a pipeline of the detector and a
non-maximum suppression stage, with the following interface.

| Item | Description |
| --- | --- |
| Input `image` | RGB image, 640 by 640 pixels |
| Input `iouThreshold` | Overlap threshold for suppression, default 0.7 |
| Input `confidenceThreshold` | Minimum detection confidence, default 0.25 |
| Output `confidence` | Confidence of each retained detection |
| Output `coordinates` | Box coordinates of each retained detection |
| Class | One class, `plank_end` |

Our evaluation in Section 5 used an overlap threshold of 0.5, so the application should set `iouThreshold` to 0.5
to reproduce those counts. We inspected the package but did not execute it, because running Core ML models requires
macOS or iOS. We have therefore not verified its predictions or its on-device latency.

## 9. Saving the results

We package the results table, the overlays, and the final weights for download. The overlays show the confidential
photographs, so we share them privately.

In [10]:
# Put the results and the final weights in one zip archive (-q is quiet, -r includes subfolders)
!zip -q -r colab_results.zip results runs/final_all/weights
# Download the archive to the local computer
files.download("colab_results.zip")

## 10. Limitations

Our evaluation rests on five photographs and 224 annotated planks, so the reported error is a rough indication and
not a reliable estimate of performance on new photographs. We annotated a single point per plank, and the training
boxes we derived from it are approximate. We trained the detector once with default hyperparameters and without
tuning, so the reported error is not an upper bound on what this model family can reach. We chose the counting
convention (every individually visible plank end on the front faces) for this task, and the original brief did not
specify it.